## Line plots
This notebook creates simple line plots to summarize the effect of three different link formation mechanisms on various network inequality metrics.
The link formation mechanisms are
- random `U`: Choose target node at random
- homophily `H`: Choose target node based on its group membership
- preferential attachment and homophily `PAH`: Choose target node based on its popularity
In addition, nodes may be limited by triadic closure, selecting only locally among their friends of friends or globally among all available nodes.

For each metric, we varying the link formation mechanisms per triadic closure and global scope, considering five combinations (`global, triadic closure`):
1. (R,R)
2. (H,R)
3. (H,H)
4. (PAH, R)
5. (PAH, PAH) 

The metrics are
- `gini`: Global degree inequality
- `ei`: Network segregation as ratio of out- and in-group links
- `mann_whitney`: Measures whether degrees of minority nodes tend to be higher than majority node degrees

As an input it takes aggregated statistics from a `.csv`-file.


### Imports and configuration

In [2]:
from typing import Optional
from itertools import product
import os

import matplotlib.pyplot as plt
import pandas as pd
from netin.models import CompoundLFM

from patch.constants import\
    PATH_STATISTICS, PATH_PLOTS,\
    N, M, F, L_HOMOPHILY, L_TAU,\
    L_LFM_LOCAL, L_LFM_GLOBAL, N_REALIZATIONS

Configuration and utility data structures to make the plotting easier.

In [3]:
COLORMAP_TC = plt.get_cmap("Oranges")
COLOR_NORM = lambda x: .15 + .7 * x

MAP_LFM_AX = {
    (CompoundLFM.UNIFORM, CompoundLFM.UNIFORM): (0, 0),
    (CompoundLFM.HOMOPHILY, CompoundLFM.UNIFORM) : (0, 1),
    (CompoundLFM.HOMOPHILY, CompoundLFM.HOMOPHILY) : (1, 1),
    (CompoundLFM.PAH, CompoundLFM.UNIFORM) : (0, 2),
    (CompoundLFM.PAH, CompoundLFM.PAH) : (1, 2),
}

plt.rcParams["font.size"] = 10


Some metadata

In [4]:
N = 5000 # Number of nodes
M = 2 # Number of new links per node
f = .3 # Fraction of the minority class

### Data
Read the data from the provided `.csv`-file.

In [ ]:
data = pd\
    .read_csv(PATH_STATISTICS)\
    .set_index(["global", "local", "tc", "h", "r"])\
    .sort_index()
data

Derive some constants (values for homophily, triadic closure and number of realizations) from the data table.

In [ ]:
TAU_VALS = data.index.get_level_values("tc").unique().sort_values()
H_VALS = data.index.get_level_values("h").unique().sort_values()
N_REAL = data.index.get_level_values("r").nunique()

print(f"homophily values: {H_VALS.values}")
assert H_VALS == L_HOMOPHILY

print(f"triadic closure values: {TAU_VALS.values}")
assert TAU_VALS == L_TAU

print(f"number of realizations: {N_REAL}")
assert N_REAL == N_REALIZATIONS

### Plotting
Create, for each metric, a grid of five plots based on the combinations of local and global link formation mechanisms.

In [7]:
def plot_metric(metric: str, data: pd.DataFrame, vneutral: Optional[float] = None):
    assert metric in data.columns,\
        f"Could not find metric `{metric}` in data columns `{data.columns}`"

    # Create plot
    fig, a_ax = plt.subplots(
        nrows=2, ncols=3,
        sharex=True, sharey=True,
        figsize=(16* (2/3),9 * (2/3)))

    # Filter data by metric column
    data_metric = data[metric]

    # Go over all possible combinations of global and local LFM
    for lfm_global, lfm_local in product(L_LFM_GLOBAL, L_LFM_LOCAL):
        if not (lfm_global, lfm_local) in data_metric.index:
            continue # Skip invalid combinations
        # Choose axis based on LFM combination
        ax = a_ax[MAP_LFM_AX[(lfm_global, lfm_local)]]

        # Filter data by LFM combination
        data_tc = data_metric.loc[(lfm_global, lfm_local)]

        # Create one line per triadic closure value
        for tc in TAU_VALS:
            # Group by h and aggregate across realizations
            data_h_aggregates = data_tc\
                .loc[tc]\
                .groupby(axis="index", level="h")\
                .aggregate(["mean", "std"]) # Copmute mean and std

            # Plot mean with std error
            ax.errorbar(
                x=H_VALS,
                y=data_h_aggregates["mean"],
                yerr=data_h_aggregates["std"],
                color=COLORMAP_TC(COLOR_NORM(tc)),
                marker="s",
                label=f"{tc}" if lfm_global == CompoundLFM.UNIFORM else None)
            if vneutral is not None:
                # Plot neutral metric line if provided
                ax.axhline(vneutral, color="black", linestyle="--")

            ax.set_title(f"{lfm_global}, {lfm_local}")

    # The bare minimum of making the plots look nice
    for ax in (a_ax[0,0], a_ax[1,1]):
        ax.set_ylabel(metric)
        ax.yaxis.set_tick_params(which='both', labelleft=True)

    for ax in a_ax[-1]:
        ax.set_xlabel("h")

    for ax in a_ax.flatten():
        ax.spines[["top", "right"]].set_visible(False)

    a_ax[0,0].set_xticks(H_VALS)
    a_ax[0,0].set_xlabel("h")
    a_ax[0,0].xaxis.set_tick_params(which='both', labelbottom=True)
    a_ax[1,0].set_visible(False)

    # Set legend and metadata in available space
    fig.legend(
        loc="lower left",
        bbox_to_anchor=(.15, .25),
        frameon=False, title="tc",
        ncols=2)
    fig.text(.07, .3, f"$N = {N}$\n$m = {M}$\n$f = {f}$")

    fig.tight_layout()
    file = os.path.join(PATH_PLOTS, f"{metric}.pdf")
    fig.savefig(file)
    print(f"Saving file to `{file}`.")

    return fig, a_ax

#### Plots per metric
Create plots for each metric.

In [ ]:
_ = plot_metric("gini", data)

In [ ]:
_ = plot_metric("ei", data, vneutral=0.)

In [ ]:
_ = plot_metric("stoch_dom", data, vneutral=0.)